# 26. Fill the gaps in sensor readings with the last known value

**Track:** sql | **Difficulty:** medium | **Tags:** forward fill, last non-null, window functions, time series

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("026-forward-fill-sensor-readings")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "readings": """
UEFSMRUEFYABFVhMFRAVABIAAEAEBQAJAQADCQcEAAQNCAACDQgABg0IAAcNCDwIAAAAAAAAAAEAAAAAAAAAFQAVggIVigIsFYAF
FRAVBhUGHBgICAAAAAAAAAAYCAEAAAAAAAAAFgAoCAgAAAAAAAAAGAgBAAAAAAAAABERAAAAgQHwgAMAAACABQEDUYjGWnRGtnV9
BnYqj94xU5TK4OamW3lnk0XZYugHnxArxdydm5P3sU6a6u+BH/ifd3Y1ZlzBwLIlUEW2v1Dh7NM9lcu09gmkOHuzkhEkw1iPGMnF
9IEfRDHVU+AAD4Q+lKinNBCzTnv0P3jfibLNAHo4zjbfMQReKhUEFYAoFaIeTBWABRUAEgAAgBQwAFqJK0MvBgAAvBPKPwEICI7k
yAUQDFIHLUQBEAw6cwpAAQgMCS67QQEIDLhLtzwBCAiLznoFEAx8bhs9ARAITsatBTAISUzWBTAMITtKQgEYCLBQTAUoCJwL0AVY
CDfkTwUQCOymVAU4CPLS8AUQCLthBQWACN6NdAVICGUheAUICBCFiAU4CFTkIgVQCNaE4gUwCP4cAgUgCBrkfAVwCDE/RwUICGCn
oAUICCOf0wU4CJz9qAWwCEWEwwVwCHqRJQVoCD2JWAVQCFzfjQUYCI7WoQUoCPkpvgU4CPsGtAVoCH2ncwUICB36ygUwCPh3+QUI
CEa9GwUIDNvwkj4B6Ai2bsEFCAgBrLwFGAgdczcFaAgP08MFEAgYgPMFIAg9AsUFCAhp22MFqAjY6GsFcAiFsAUF0AgyangFkAiA
vcEFGAj+lW4FKAgHQ54FoAgDibIFEAgyeJ8FMAgerI8FKAgKWewFWAiWZscFcAgN9s0FIAg3a+MFCAi+d1MFyAjHqxYFCAgMvXUF
SAi+8L8FcAgpvUgFGAgGGG0FuAitOv4FUAjVS4oFIAggiYUFIAjkq+kFCAgHyjEFQAhruFkFKAhihJYFCAiDTHwFgAhjNlsFSAjh
lZsFCAiKlSIFSAg+Ox0FsAizZpoFCAiIP8AFMAj7fyAFQAhoKZ8FMAg0zgEFCAjqUPIFIAhaApgFEAgy8QsFgAhzOlgFUAhrP+0F
EAiR7O8FQAhPaksFCAiT1wwF+Ah3AmsFqAgn4FIFcAiqMtcFCAhOuIYFQAhJxUIFCAh3if4FGAgXzi4FQAjCuNIFOAjLcykFSAiX
GIwFIAh/hGkFiAiTUHkFIAgGnwAFGAiluP8FGAz+Dts8QRAIqqtDBSAIRjaIBVAIaLAyBSgI5CRWBRAIfSDgBXgIlHuqBQgIxzKq
CWgEkfgFSAjnwTcFEAjef00FIAjw59MFWAidvZQFGAgs05YFUAhXc90FaAhs8bElKAi0GF8FIAjfuKUlGAi/KRgFGAjz/SEFWAiQ
OisFgAiuc1YFQAjSvM8FEAhiCyoFKAh89a4FUAgvYlEFEAjtWBkFIAjhHC8FMAgVaqUFiAi/ooQFGAhX+nAFcAjZE50FOAh/C/0F
OAj48GUFcAg3XbwFwAjZmjAFKAichDwFKAjvQzYFUAjQ39kFEAiNq3AFMAikBjsFCAhDp80FCAi5hA8FyAgMNuIFeAgmLo4FYAgY
B4cFCAhRVWgFSAhx5PUFEAgEO3cFMAgeJfwFQAhpYvcFIAj2E3AFGAir1nQFkAiw198FkAjHJIMFEAiFot4FYAjbdyYFSAjNUB8F
CAjV0h0FOAjt36wFCAirXQgFCAgkygQFCAjW/U4lCAiIuCwFcAjZjAkFUAiQwb4FIAjcKesFSAi8ml0FiAjBBg4F8AiCmrcFIAg4
lhQFIAgv270FSAhM25AFSAhAkX8FSAgEtOMFCAhOPxoFMAgMRAkFuAhmxRUFMAiLR+cFIAiOXTUFWAgmp/oFGAhgLjQFKAhdGOYF
OAir5JsFEAjpF5oFIAjw9foFkAjFziAFCAgNbzoFKAinlfUFUAjnsxAFCAgK4H8FoAhLKcwFOAjyS10FCAjeBuEFcAiUidEFWAhX
7EkFMAi5C6MFEAgV8TgFIAhJ02kFEAjsLegFEAiZbu4FKAhImhEFEAi29VQFYAgbD64FGAjEHFwFEAiGYsoFOAhAGBMFmAjCP2YF
EAh8fEIFCAj12hcFMAgBJSklMAhjve4FOAiap0YFCAgpNrUFIAjElcgFsAiozk0FGAjvvKIFMAjCMT8FIAh07BwFIAiwybgFEAgu
KfkFCAiANi4FoAjTbpQFMAhUXY8FKAh6GLkFSAgsTAMFIAhdkVIFGAg0VZUFGAiqJLBFSAh0c7AFuAjYYdgFQAhVlucFIAggAvIF
YAilP5MleAjKOtEFIAgjGEAFUAg7rGIF4AjQZm0FCAiFKXIFGAih8OwFWAiUAj4FWAgjkawFEAiW3zMFQAiz3wYFKAjT9ScFWAhS
gJkFEAioR7oFoAgSYn4FOAgaXekFOAhlE1EFiAhDLmEFMAhlmuQFMAh5ZvQFKAg1gMYFWAgbHdUFiAjQWEYFMAitwZEFyAiiorEF
EAifmooFEAifE/cFaAhAn6YFMAhMYiQFWAjwbmcFWAgDAh8FMAhmTKkFYAhuRxQFEAgkUZgFMAghtLYFQAgblkEFIAgS2+oFIAiu
7MIFiAg7Jc8FSAjizvMFWAgHUcUFCAhuzqcFGAixAhEFiAj16D4FUAg4D4EFUAgxuLMF8Ah3e9cFCAjNyYsFGAhRztQFWAiiKUUF
cAhf9dsFEAjBjaEFWAh2UKYFEAjKwWQFEAiRc4MFYAiCEyQFQAgVeMwFGAhxXWIFKAjIXdsFSAjqyV4FEAhuVTsFiAjzhLUFOAj1
YasFqAhgIA0F2AhLsF8FGAgAc2QFUAinHIkFQAiTyeUFqAjcolcFSAg6+p0FaAhP8d4FIAgpRNwFWAi5/XsFSAgJpycFQAgPTDAF
MAiisNgFIAjFR40FEAh532AFIAiZ9YEFQDjnOqRALwYAADUHWkMvBgAVABXiBRXsBSwVgAUVEBUGFQYcGAgAN2vjRC8GABgIAAdD
njwvBgAWACgIADdr40QvBgAYCAAHQ548LwYAEREAAADxAvRwAQMAAACABQEJUQACCBhAoICBAwgSKFjAoIGDBxAiSJhAoYKFCxgy
aNjAoYOHDyBCiBhBooSJEyhSqFjBooWLFzBiyJhBo4aNGzhy6NjBo4ePH0CCCBlCpIiRI0iSKFnCpImTJ1CiSJlCpYqVK1iyaNnC
pYuXL2DCiBlDpoyZM2jSqFnDpo2bN3DiyJlDp46dO3jy6NnDp4+fP4ACCRpEqJChQ4gSKVrEqJGjR5AiSZpEqZKlS5gyadrEqZOn
T6BCiRpFqpSpU6hSqVrFqpWrV7BiyZpFq5atW7hy6drFq5evX8CCCRtGrJixY8iSKVvGrJmzZ9CiSZtGrZq1a9iyadvGrZu3b+DC
iRtHrpy5c+jSqVvHrp27d/DiyZtHr569e/jy6dvHr5+/fwADChxIsKDBgwgTKlzIsKHDhxAjSpxIsaLFixgzatzIsaPHjyBDihxJ
sqTJkyhTqlzJsqXLlzBjypxJs6bNmzhz6tzJs6fPnxUEFcAHFZAETBV4FQASAADgAyCamZmZmRk4QAAFAQQ0QAUQDJk5QDMBAQSz
NA0IADcJCCQzNEDNzMzMzEw5CTAEgDMJGASzOQkYBMw0CRgEADUJGAwzOEBmAQEE5jENCA0gBIA3DSgReAA6DXgAMQlIAEwROAA5
DTgAOA0gADQJCAQZNQ1AADkNaAAyDRgRmBE4ADMNKAAyCUgEZjoNCAA4CTgAsw1wAMwNCABMEaARCAAyDSgAMwmYBMwzCWgAmRGQ
ADINGAAyCWgEgDgNEA0gABkNMAAZDSAEADMJgATmNwkIBGY5CVAEGTQNEAA0DUAANwkYAJkxYA1YAEwNGAAZEcgNIAAzMYAAOQ1I
JDVAZmZmZmZmN0AVABXkAxXsAywVgAUVEBUGFQYcGAhmZmZmZmY6QBgImpmZmZmZMUAWnAEoCGZmZmZmZjpAGAiamZmZmZkxQBER
AAAA8gHw8TIAAAAJ3T765xYBCf57T6EQAQP2GAErLpt/Z/Pf6bZzP703f7V729/6N2ufFgEF+vcUAQX+CAY/QCAMRGEcSKIsBtMc
zgMcDzI8UCRNVFNVVtUEF5YNmlY8x8Z1wpJ4VfYJFwEq3pCE4pMII0UyVKKVgPZplVmWwYkyzgd4DMMdHAKkRmMkV1KkKglIojE0
q8C2QuU+L5eZWqhAIQIbaGmUIQAjbIAoBezEpexqn5e9JQNXxzpLGDXHJd3AX6VOtUcgRkUzRrWoZLEtiVVOoc3dMQMym8diUNME
rvpYuDxw5fA2eQpIyTfpcGjsHwkAAAAAFQQZTDUAGAZzY2hlbWEVBgAVBCUCGAlkZXZpY2VfaWQAFQQlAhgKcmVhZGluZ190cyUU
TIwSHCwAAAAAABUKJQIYC3RlbXBlcmF0dXJlABaABRkcGTwmABwVBBk1AAYQGRgJZGV2aWNlX2lkFQIWgAUWpAQWhAQmfiYIHBgI
CAAAAAAAAAAYCAEAAAAAAAAAFgAoCAgAAAAAAAAAGAgBAAAAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgCABQAAACYAHBUE
GTUABhAZGApyZWFkaW5nX3RzFQIWgAUWiC8WtCUm0CImjAQcGAgAN2vjRC8GABgIAAdDnjwvBgAWACgIADdr40QvBgAYCAAHQ548
LwYAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAIAFAAAAJgAcFQoZNQAGEBkYC3RlbXBlcmF0dXJlFQIWgAUWygwWogkm8C0mwCkc
GAhmZmZmZmY6QBgImpmZmZmZMUAWnAEoCGZmZmZmZjpAGAiamZmZmZkxQBERABksFQQVABUCABUAFRAVAgA8KQYZJpwB5AMAAAAW
9j8WgAUmCBbaMgAZHBgMQVJST1c6c2NoZW1hGMwCLy8vLy8vQUFBQUFRQUFBQUFBQUtBQXdBQmdBRkFBZ0FDZ0FBQUFBQkJBQU1B
QUFBQ0FBSUFBQUFCQUFJQUFBQUJBQUFBQU1BQUFDRUFBQUFPQUFBQUFRQUFBQ1kvLy8vQUFBQkF4QUFBQUFjQUFBQUJBQUFBQUFB
QUFBTEFBQUFkR1Z0Y0dWeVlYUjFjbVVBenYvLy93QUFBZ0RJLy8vL0FBQUJDaEFBQUFBa0FBQUFCQUFBQUFBQUFBQUtBQUFBY21W
aFpHbHVaMTkwY3dBQUFBQUdBQWdBQmdBR0FBQUFBQUFDQUJBQUZBQUlBQVlBQndBTUFBQUFFQUFRQUFBQUFBQUJBaEFBQUFBa0FB
QUFCQUFBQUFBQUFBQUpBQUFBWkdWMmFXTmxYMmxrQUFBQUNBQU1BQWdBQndBSUFBQUFBQUFBQVVBQUFBQT0AGCBwYXJxdWV0LWNw
cC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRk8HAAAHAAAHAAAAEMDAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "026-forward-fill-sensor-readings"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

readings = spark.read.parquet((_data_dir / "readings.parquet").resolve().as_posix())
readings.createOrReplaceTempView("readings")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'device_id': 'b7c88243645dd1b0b631bbb8c6528e2702943056f0209e2145e83ac09da8527b',
                         'filled_temperature': '49c0da4b384fe0918db8dc63557f02c4592d65bbf4f5214aeecf095101cc4ef2',
                         'reading_ts': '31492d2513bd0f4690ac22afab74a4b994d8bf4ed594b1c90e0d7355fb6c9031',
                         'temperature': '687220becdfb485b526dc20ccc3c6cc691fd3853131f559921f5f90b75479eca'},
 'columns': [['device_id', 'bigint'],
             ['filled_temperature', 'double'],
             ['reading_ts', 'timestamp_ntz'],
             ['temperature', 'double']],
 'fingerprint': '48b829cbafd7be49db0e711751b3a35bdc314ee2b15ac61271dace433abfdad1',
 'order_matters': False,
 'row_count': 320}

check = make_check(EXPECTED)

## The situation

Cold-storage sensors report a temperature every 15 minutes. Sometimes a sensor fails
to report and the reading arrives as NULL. A monitoring chart needs a value at every
point, so each missing reading should show the last temperature the device did
report.

A first attempt used the previous row's value. It works for a single missing
reading and leaves holes wherever two or more are missing in a row.

## Input tables

### `readings`

One row per device every 15 minutes. A device never has two rows with the same
timestamp.

| Column | Type | Description |
|---|---|---|
| `device_id` | BIGINT | The sensor. |
| `reading_ts` | TIMESTAMP_NTZ | When the reading was due. |
| `temperature` | DOUBLE | Degrees Celsius. NULL if the sensor did not report. |

## Requirement

Return every row of `readings` with one more column, `filled_temperature`.

- If `temperature` is present, `filled_temperature` is the same value.
- If it is NULL, `filled_temperature` is the device's most recent earlier
  temperature that is not NULL, however many readings back that is.
- Values are never taken from another device.
- Values are never taken from a later reading. If a device has no earlier reading
  at all, `filled_temperature` is NULL.

## Expected output

One row per row of `readings`.

| Column | Type | Description |
|---|---|---|
| `device_id` | BIGINT | |
| `reading_ts` | TIMESTAMP_NTZ | |
| `temperature` | DOUBLE | The original value, NULL included. |
| `filled_temperature` | DOUBLE | The reading, or the last known one before it. |

In [ ]:
readings.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `readings`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)